# Customer Lifetime Value Prediction Using Statistical Machine Learning

Reproducible demonstration of customer-level 12-month value prediction, statistical testing, regression evaluation, and value-tier classification.

> **Data caveat:** this notebook generates synthetic data. Metrics are illustrative and must not be interpreted as evidence of real-world business performance.

## Workflow
1. Generate a synthetic customer dataset with a fixed random seed.
2. Inspect and explore customer characteristics.
3. Compare loyalty segments using Welch's t-test.
4. Train preprocessing pipelines and compare regression baselines.
5. Evaluate held-out performance and cross-validation.
6. Inspect permutation importance and classify CLV tiers.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Generate synthetic customer data

Replace this generator with an approved, representative customer dataset for real-world evaluation. Ensure predictors are available at the prediction date and the target represents a subsequent future window.

In [ ]:
n = 2500
age = rng.integers(18, 76, n)
tenure = rng.integers(1, 97, n)
orders = np.clip(rng.poisson(5 + tenure / 20, n), 1, 35)
aov = np.clip(rng.lognormal(3.55, 0.48, n), 8, 500)
spend = orders * aov * rng.normal(1.0, 0.12, n)
recency = np.clip(rng.gamma(2.0, 35, n), 0, 365).round().astype(int)
returns = rng.beta(1.4, 12, n)
engagement = rng.beta(2.2, 2.0, n)
loyalty = rng.binomial(1, 0.42, n)
channel = rng.choice(['Organic','Paid Search','Social','Referral','Email'], n, p=[.24,.25,.20,.16,.15])
region = rng.choice(['North','South','East','West'], n)
retention = np.clip(0.25 + .55*engagement + .30*loyalty + .18*np.log1p(tenure/12) - .0018*recency - .65*returns, .08, 1.25)
future_value = np.clip(spend * retention * rng.lognormal(.05, .38, n) + orders*aov*.12, 0, None)
df = pd.DataFrame({'customer_id':[f'C{i:05d}' for i in range(1,n+1)], 'age':age, 'tenure_months':tenure, 'orders_12m':orders, 'avg_order_value':aov.round(2), 'spend_12m':spend.round(2), 'days_since_last_order':recency, 'returns_rate':returns.round(4), 'email_engagement':engagement.round(4), 'loyalty_member':np.where(loyalty==1,'Yes','No'), 'acquisition_channel':channel, 'region':region, 'future_12m_value':future_value.round(2)})
for col in ['age','avg_order_value','email_engagement']:
    df.loc[rng.choice(df.index, int(.01*n), replace=False), col] = np.nan
print('Shape:', df.shape)
display(df.head())
display(df.isna().sum().to_frame('Missing values'))

## 2. Exploratory analysis and statistical test

In [ ]:
display(df.describe(numeric_only=True).T.round(2))
sns.histplot(df['future_12m_value'], kde=True)
plt.title('Distribution of synthetic future 12-month customer value')
plt.xlabel('Future 12-month value')
plt.tight_layout(); plt.show()

loyal = df.loc[df.loyalty_member == 'Yes', 'future_12m_value']
non_loyal = df.loc[df.loyalty_member == 'No', 'future_12m_value']
t_stat, p_value = stats.ttest_ind(loyal, non_loyal, equal_var=False)
print(f'Welch t-statistic: {t_stat:.3f}; p-value: {p_value:.6g}')
print(f'Mean difference (loyal - non-loyal): {loyal.mean()-non_loyal.mean():.2f}')
print('Interpretation applies only to this synthetic data-generating process.')

## 3. Preprocessing and regression models

Imputation, scaling, and encoding are fitted within scikit-learn pipelines to avoid fitting preprocessing on the held-out test set.

In [ ]:
target = 'future_12m_value'
X = df.drop(columns=['customer_id', target])
y = df[target].astype(float)
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()
preprocessor = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols), ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=RANDOM_STATE)
models = {'Dummy Mean Baseline': DummyRegressor(strategy='mean'), 'Multiple Linear Regression': LinearRegression(), 'Ridge Regression': Ridge(alpha=1.0), 'Random Forest': RandomForestRegressor(n_estimators=250, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)}
fitted, rows = {}, []
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    pred = np.clip(pipe.predict(X_test), 0, None)
    fitted[name] = pipe
    rows.append({'Model':name, 'MAE':mean_absolute_error(y_test,pred), 'RMSE':np.sqrt(mean_squared_error(y_test,pred)), 'R2':r2_score(y_test,pred)})
results = pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)
display(results.round(3))
best_name = results.loc[0, 'Model']
best_model = fitted[best_name]
print('Best model by held-out RMSE:', best_name)

## 4. Cross-validation and feature importance

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', estimator)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
    cv_rows.append({'Model':name, 'CV RMSE mean':-scores.mean(), 'CV RMSE std':scores.std()})
display(pd.DataFrame(cv_rows).sort_values('CV RMSE mean').round(3))

perm = permutation_importance(best_model, X_test, y_test, n_repeats=8, random_state=RANDOM_STATE, scoring='neg_root_mean_squared_error', n_jobs=-1)
importance = pd.DataFrame({'Feature':X_test.columns, 'Importance mean':perm.importances_mean, 'Importance std':perm.importances_std}).sort_values('Importance mean', ascending=False)
display(importance.head(12).round(4))
sns.barplot(data=importance.head(12), x='Importance mean', y='Feature')
plt.title(f'Permutation importance — {best_name}')
plt.tight_layout(); plt.show()

## 5. Customer value tier classification

Tier boundaries are derived from the training target distribution. Classification metrics are exploratory and should be re-evaluated on real data.

In [ ]:
q1, q2, q3 = y_train.quantile([.25,.50,.75]).to_numpy()
edges = [-np.inf, q1, q2, q3, np.inf]
labels = ['Low','Medium-Low','Medium-High','High']
y_train_tier = pd.cut(y_train, bins=edges, labels=labels, include_lowest=True)
y_test_tier = pd.cut(y_test, bins=edges, labels=labels, include_lowest=True)
classifier = Pipeline([('preprocess', preprocessor), ('model', RandomForestClassifier(n_estimators=250, min_samples_leaf=3, random_state=RANDOM_STATE, class_weight='balanced'))])
classifier.fit(X_train, y_train_tier)
tier_pred = classifier.predict(X_test)
print(classification_report(y_test_tier, tier_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test_tier, tier_pred, labels=labels, xticks_rotation=30)
plt.tight_layout(); plt.show()

## 6. Summary and responsible use

This notebook demonstrates an end-to-end workflow on synthetic data. Before operational use, replace the generator with governed real data, validate with a time-based holdout, examine segment-level errors and prediction uncertainty, and monitor drift and business outcomes. Do not treat synthetic-data metrics as production evidence.